# StageBridge: Hierarchical Transformer-Based LUAD Stage Progression

**Research notebook** — single entry point for the full StageBridge pipeline.

This notebook follows a 12-step protocol with explicit QC gates at each stage:

| Step | Purpose | Key Output |
|------|---------|------------|
| 1 | Configure run | Dataset, context mode, training profile |
| 2 | Validate environment | Asset paths, data availability |
| 3 | Dataset preview | snRNA PCA/UMAP/t-SNE, Visium spatial, WES landscape |
| 4 | HLCA reference alignment | Latent mapping, alignment gate |
| 5 | Spatial provider rebuild | Tangram/TACCO/DestVI comparison |
| 6 | Provider selection | Hybrid benchmark, winner |
| 7 | Typed token construction | Epithelial/stromal/immune/vascular niche tokens |
| 8 | Transformer diagnostics | Attention patterns, context encoder inspection |
| 9 | Transition fit | Schrödinger bridge training on active edge |
| 10 | Evaluation | Held-out metrics, calibration, context sensitivity |
| 11 | EA-MIST lesion benchmark | Lesion-level Set Transformer results |
| 12 | Results writeout | Registry, artifacts, publication figures |

Each step produces a gate status. Proceed only if the gate passes.

In [ ]:
# --- Step 1: Configure Run ---
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Run configuration
DATASET = "luad_evo"
CONTEXT_MODE = "typed_hierarchical_transformer"
ACTIVE_EDGE = "AAH->AIS"
TRAINING_PROFILE = "medium"
WES_ENABLED = True

# Paths
RUN_ROOT = Path("outputs/scratch/stagebridge_v1/eamist_benchmark")
REPORT_ROOT = Path("reports")
DATA_ROOT = Path("/mnt/e/StageBridge_data")

print(f"Dataset:       {DATASET}")
print(f"Context mode:  {CONTEXT_MODE}")
print(f"Active edge:   {ACTIVE_EDGE}")
print(f"Training:      {TRAINING_PROFILE}")
print(f"WES:           {'enabled' if WES_ENABLED else 'disabled'}")
print(f"Run root:      {RUN_ROOT}")
print(f"Data root:     {DATA_ROOT}")

## Step 2: Validate Environment and Asset Paths

Check that all required data assets exist before proceeding.

In [ ]:
# --- Step 2: Validate Environment ---
import torch

assets = {
    "snRNA merged": DATA_ROOT / "processed" / "anndata" / "snrna_merged.h5ad",
    "Visium merged": DATA_ROOT / "processed" / "anndata" / "spatial_merged.h5ad",
    "WES features": DATA_ROOT / "processed" / "features" / "wes_features.parquet",
}

print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"GPU VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")
print()

all_ok = True
for name, path in assets.items():
    status = "OK" if path.exists() else "MISSING"
    if status == "MISSING":
        all_ok = False
    print(f"  [{status}] {name}: {path}")

gate = "PASS" if all_ok else "FAIL"
print(f"\nEnvironment gate: {gate}")

## Step 3: Dataset Preprocessing and Cohort Preview

Visualize the raw snRNA-seq, spatial transcriptomic, and WES data before any modeling.

**Figures produced:**
- PCA (with explained variance %), UMAP, t-SNE colored by stage
- Visium spot layout with epithelial/stromal/immune/vascular marker expression
- WES tumor mutation burden, oncoprint, and mutation frequency by stage

In [ ]:
# --- Step 3: Dataset Preview ---
import matplotlib.pyplot as plt
from stagebridge.notebook_api import compose_config, run_data_preprocessing
from stagebridge.viz.research_frontend import (
    configure_research_style,
    plot_snrna_preprocessing_frontend,
    plot_spatial_preprocessing_frontend,
    plot_wes_preprocessing_frontend,
)

configure_research_style()

cfg = compose_config(overrides=[
    f"context_model={CONTEXT_MODE}",
    f"data.active_edge={ACTIVE_EDGE}",
])

data_output = run_data_preprocessing(cfg)

# snRNA: PCA with variance %, UMAP, t-SNE, cell counts
fig_snrna = plot_snrna_preprocessing_frontend(data_output)
display(fig_snrna); plt.close(fig_snrna)

# Visium: spatial spot layout, marker genes
fig_spatial = plot_spatial_preprocessing_frontend(data_output)
display(fig_spatial); plt.close(fig_spatial)

# WES: TMB, oncoprint, mutation frequency, stage-stratified mutations
fig_wes = plot_wes_preprocessing_frontend(data_output)
display(fig_wes); plt.close(fig_wes)

print(f"\nsnRNA: {data_output['snrna']['n_cells']:,} cells, {data_output['snrna']['n_genes']:,} genes")
print(f"Visium: {data_output['spatial']['n_spots']:,} spots")
print(f"WES: {data_output['wes']['n_rows']} samples")

## Step 4: HLCA Reference Latent Mapping

Align the LUAD cohort into the HLCA latent space. The alignment gate checks:
- Stage probe accuracy (does the latent preserve stage identity?)
- Donor leakage (is the latent batch-confounded?)
- Gene overlap and label agreement with HLCA

In [ ]:
# --- Step 4: HLCA Reference Alignment ---
import matplotlib.pyplot as plt
from stagebridge.notebook_api import run_reference_backend
from stagebridge.viz.research_frontend import plot_reference_frontend

reference_output = run_reference_backend(cfg, data_output)

fig_ref = plot_reference_frontend(reference_output)
display(fig_ref); plt.close(fig_ref)

gate_status = reference_output["reference"]["diagnostics"].get("alignment_gate", {}).get("status", "n/a")
print(f"\nAlignment gate: {gate_status}")

## Step 5-6: Spatial Provider Rebuild and Selection

Rebuild Tangram, TACCO, and DestVI spatial mappings from scratch.
Run hybrid benchmark scoring (mapping QC + downstream performance) to select the best provider.

In [ ]:
# --- Steps 5-6: Spatial Provider Rebuild and Selection ---
import matplotlib.pyplot as plt
from stagebridge.notebook_api import run_spatial_providers, run_provider_benchmark
from stagebridge.viz.research_frontend import (
    plot_spatial_provider_comparison_frontend,
    plot_spatial_provider_maps_frontend,
    plot_provider_benchmark_frontend,
)

provider_outputs = run_spatial_providers(cfg, data_output, reference_output)

fig_comparison = plot_spatial_provider_comparison_frontend(provider_outputs)
display(fig_comparison); plt.close(fig_comparison)

fig_maps = plot_spatial_provider_maps_frontend(provider_outputs)
display(fig_maps); plt.close(fig_maps)

benchmark_output = run_provider_benchmark(cfg, provider_outputs, data_output)
fig_bench = plot_provider_benchmark_frontend(benchmark_output)
display(fig_bench); plt.close(fig_bench)

selected = (benchmark_output.get("benchmark") or {}).get("selected_provider", "n/a")
print(f"\nSelected provider: {selected}")

## Step 7-8: Context Encoder and Transformer Diagnostics

Build typed niche tokens from the selected spatial provider, then inspect
the hierarchical transformer's attention patterns.

**Key question:** Does the transformer attend differentially to biological groups
across stages? The fusion attention heatmap should show stage-dependent weighting.

In [ ]:
# --- Steps 7-8: Context Encoder and Transformer Diagnostics ---
import matplotlib.pyplot as plt
from stagebridge.notebook_api import run_context_branch
from stagebridge.viz.research_frontend import (
    plot_context_frontend,
    plot_transformer_attention_frontend,
    plot_spatial_mapping_frontend,
)

context_output = run_context_branch(cfg, data_output, reference_output, benchmark_output)

# Spatial mapping summary
fig_mapping = plot_spatial_mapping_frontend(context_output)
display(fig_mapping); plt.close(fig_mapping)

# Typed niche context: stage-wise composition, dominant groups, spatial map
fig_context = plot_context_frontend(context_output)
display(fig_context); plt.close(fig_context)

# Transformer attention diagnostics: fusion heatmap, group profiles, relations
fig_attn = plot_transformer_attention_frontend(context_output)
display(fig_attn); plt.close(fig_attn)

print(f"Context mode: {context_output['context_model'].get('mode', 'n/a')}")
print(f"Context dim:  {context_output['context_model'].get('example_context_dim', 'n/a')}")

## Step 9-10: Transition Fit and Evaluation

Train the Schrödinger bridge on the active edge with population context conditioning.
Evaluate on held-out donors using Sinkhorn distance, MMD, classifier AUC, and calibration.

**Figures produced:**
- Source/predicted/target PCA manifold (with variance %)
- Training loss curves (total, drift, diffusion)
- Macroflow heatmap (source cluster to predicted cluster)
- Biological insight: typed niche shift profiles

In [ ]:
# --- Steps 9-10: Transition Fit and Evaluation ---
import matplotlib.pyplot as plt
from stagebridge.notebook_api import run_transition_fit, run_evaluation
from stagebridge.viz.research_frontend import (
    plot_transition_frontend,
    plot_biological_insight_frontend,
)

transition_output = run_transition_fit(cfg, data_output, reference_output, context_output)
evaluation_output = run_evaluation(cfg, transition_output)

# Transition dynamics: PCA manifold, training curves, macroflow
fig_transition = plot_transition_frontend(transition_output, evaluation_output)
display(fig_transition); plt.close(fig_transition)

# Biological insight: typed niche shift by group
fig_biology = plot_biological_insight_frontend(evaluation_output)
display(fig_biology); plt.close(fig_biology)

heldout = evaluation_output["heldout_metrics"]
print(f"\nHeld-out metrics ({ACTIVE_EDGE}):")
print(f"  Sinkhorn:        {heldout['sinkhorn']:.4f}")
print(f"  MMD-RBF:         {heldout['mmd_rbf']:.4f}")
print(f"  Classifier AUC:  {heldout['classifier_auc']:.4f}")
print(f"  Direction cos:   {heldout['direction_cosine']:.4f}")
print(f"  Calibration err: {evaluation_output['calibration']['mean_abs_shift_error']:.4f}")

## Step 11: EA-MIST Lesion-Level Benchmark

The EA-MIST architecture treats lesions as bags of spatial niche prototypes.
This step loads and displays the lesion-level benchmark results.

In [ ]:
# --- Step 11: EA-MIST Lesion Benchmark ---
benchmark_summary = RUN_ROOT / "benchmark_summary.csv"
tables_root = REPORT_ROOT / "tables" / "eamist"
figures_root = REPORT_ROOT / "figures" / "eamist"

if benchmark_summary.exists():
    benchmark_df = pd.read_csv(benchmark_summary)
    display(Markdown("### Benchmark results (top models per edge)"))
    display(benchmark_df.sort_values(["edge_label", "auroc"], ascending=[True, False]).head(20))
else:
    display(Markdown(f"_Benchmark summary not found at `{benchmark_summary}`._\n\nRun the EA-MIST pipeline first."))

# Embedding diagnostics figure
embedding_fig = figures_root / "figure2_embedding_diagnostics.png"
if embedding_fig.exists():
    display(Markdown(f"### Embedding diagnostics\n\n![embedding diagnostics]({embedding_fig.as_posix()})"))

# Prototype interpretation figure
prototype_fig = figures_root / "figure4_prototypes_attention.png"
if prototype_fig.exists():
    display(Markdown(f"### Prototype interpretation\n\n![prototypes]({prototype_fig.as_posix()})"))

# Ablation figure
ablation_fig = figures_root / "figure5_ablations.png"
if ablation_fig.exists():
    display(Markdown(f"### Ablation study\n\n![ablations]({ablation_fig.as_posix()})"))

## Step 12: Results Summary

Key findings from this run:

| Edge | Best Context Mode | Sinkhorn | AUC | Key Insight |
|------|-------------------|----------|-----|-------------|
| AAH→AIS | Set Transformer / RNA-only | TBD | TBD | Context helps less at early initiation |
| AIS→MIA | Set Transformer | TBD | TBD | Context captures microenvironmental shift |

### Architecture summary

The hierarchical transformer context encoder (`hidden_dim=192`, `8 heads`, `24 inducing points`, `4 summary tokens/group`, `8 fusion queries`) processes typed spatial niche tokens through three levels of abstraction before conditioning the transition model.

### Limitations

- Donor-held-out cross-validation relies on limited patient counts per stage
- HLCA reference alignment is imperfect (weak pass) — latent quality affects all downstream
- WES integration is regularization only, not a primary signal source
- PHATE embedding requires `phate` package (graceful fallback to UMAP)

In [ ]:
# --- Step 12: Results Writeout ---
from stagebridge.notebook_api import write_results_registry

results = write_results_registry(
    cfg=cfg,
    data_output=data_output,
    reference_output=reference_output,
    context_output=context_output,
    transition_output=transition_output,
    evaluation_output=evaluation_output,
)

print(f"Results written to: {results.get('registry_path', 'n/a')}")
print(f"Run ID: {results.get('run_id', 'n/a')}")
print("\nPipeline complete.")